# **Question 4: Building a Custom Dataset for Image Colorization - SOLUTION**

---

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
cifar_data = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True,
    transform=transforms.ToTensor()
)

print(f"CIFAR-10 loaded: {len(cifar_data)} images")
print(f"Image shape: {cifar_data[0][0].shape}")

In [ ]:
class ColorizationDataset(Dataset):
    """
    A dataset for image colorization.
    Returns (grayscale_image, color_image) pairs.
    """
    
    def __init__(self, cifar_dataset):
        self.dataset = cifar_dataset
    
    def __len__(self):
        return len(self.dataset)
    
    def rgb_to_grayscale(self, img):
        """
        Convert an RGB image to grayscale.
        Gray = 0.299 * R + 0.587 * G + 0.114 * B
        """
        # img shape: (3, H, W)
        r, g, b = img[0], img[1], img[2]
        gray = 0.299 * r + 0.587 * g + 0.114 * b
        # Add channel dimension: (H, W) -> (1, H, W)
        gray = gray.unsqueeze(0)
        return gray
    
    def __getitem__(self, idx):
        # Get color image (ignore label)
        color_img, _ = self.dataset[idx]
        
        # Convert to grayscale
        gray_img = self.rgb_to_grayscale(color_img)
        
        return gray_img, color_img

In [ ]:
# Test your implementation
colorization_dataset = ColorizationDataset(cifar_data)

# Get a sample
gray_img, color_img = colorization_dataset[0]

print(f"Grayscale image shape: {gray_img.shape}")
print(f"Color image shape: {color_img.shape}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(gray_img.squeeze(), cmap='gray')
axes[0].set_title('Grayscale (Input)')
axes[0].axis('off')
axes[1].imshow(color_img.permute(1, 2, 0))
axes[1].set_title('Color (Target)')
axes[1].axis('off')
plt.show()

In [ ]:
# Test with DataLoader
dataloader = DataLoader(colorization_dataset, batch_size=8, shuffle=True)

gray_batch, color_batch = next(iter(dataloader))
print(f"Batch grayscale shape: {gray_batch.shape}")
print(f"Batch color shape: {color_batch.shape}")